In [ ]:
# ============================================================
# Section 1 — Imports and Load Data
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from scipy.optimize import minimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sa1_merged_eda.csv', parse_dates=['SETTLEMENTDATE'])
df = df.sort_values('SETTLEMENTDATE').reset_index(drop=True)

print(f'Rows: {len(df):,}')
print(f'Date range: {df.SETTLEMENTDATE.min()} to {df.SETTLEMENTDATE.max()}')

In [ ]:
# ============================================================
# Section 2 — Prepare the series
# ============================================================

# Drop rows missing the columns we need
mdf = df[['SETTLEMENTDATE','RRP','CURTAILMENT_MW','TOTALDEMAND']].dropna().reset_index(drop=True)

# Subtract the average price for each 30-min slot of the day
# This removes the daily cycle so GARCH models genuine surprises
mdf['time_slot'] = mdf['SETTLEMENTDATE'].dt.hour * 2 + (mdf['SETTLEMENTDATE'].dt.minute >= 30).astype(int)
seasonal_mean = mdf.groupby('time_slot')['RRP'].mean()
mdf['RRP_seasonal_mean'] = mdf['time_slot'].map(seasonal_mean)
mdf['RRP_adjusted'] = mdf['RRP'] - mdf['RRP_seasonal_mean']

# ADF test — confirm the series is stationary
result = adfuller(mdf['RRP_adjusted'].dropna())
print(f'ADF p-value: {result[1]:.6f}')
print('Stationary' if result[1] < 0.05 else 'NOT stationary — needs differencing')

# Train/test split — 2023 trains, 2024 tests
train = mdf[mdf['SETTLEMENTDATE'] < '2024-01-01'].reset_index(drop=True)
test  = mdf[mdf['SETTLEMENTDATE'] >= '2024-01-01'].reset_index(drop=True)

print(f'Train: {len(train):,} rows')
print(f'Test:  {len(test):,} rows')